# Tersoff model diagnostics: the ablation, drawn

The Tersoff model (`docs/fff_tersoff.md`, `src/ff/tersoff/`) against the pairing model it ablates
and the Q-Chem scans of `qchem_roundtrip/pairing_scans`. All code lives in
`notebooks/tersoff_plots.py` (autoreloaded; it re-exports `pairing_plots`), this notebook sets the
knobs and draws the figures into `notebooks/figures/tersoff_*`.

**What is compared.** Three models at their priors unless a checkpoint is given: the Tersoff
model with the `waterfill` rule (the default), the same with the `rebo` rule (the ablation the
question asked for), and the pairing model. Same coupling, same heads, same priors, same assembly;
only how `p_ij` comes out of `J_ij` differs. Every model is evaluated on exactly the frames Q-Chem
labeled, plus a hydrogen-bonded water dimer along the O-O distance (the discriminator) that has no
labels and needs none.

**What to look for.** On intact water the curves of `tersoff` and `pairing` should lie on top of
each other; `rebo` fails there (its raw orders above one half on the 1-3 and hydrogen-bond pairs
are shared linearly and eat the covalent bonds). On the proton-transfer scans the two bond-order
models agree except within ~0.15 Å of the midpoint, where the per-atom filling leaves the shared
proton under capacity (the dotted sum in the state figure) and the closed-form charge hand-over is
sharper than the solve's: the cusp on the `tersoff` curve is that. Whether training closes it is
the question the branch exists to answer.

The RKS caveat of the pairing notebook holds: the stretch labels are physical inside ~1.8 Å only.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))

import tersoff_plots as tp

In [ ]:
# ---- Knobs ---------------------------------------------------------------------------------
#: Checkpoints (`film.model: tersoff` / `pairing`), or None for the models at their priors.
TERSOFF_CHECKPOINT = None
PAIRING_CHECKPOINT = None
#: Also draw the rebo rule at the priors (it fails intact water; worth one look).
WITH_REBO = True
#: Frames of each monomer AIMD set used for the fragment-energy parity.
MONOMER_FRAMES = 200

In [ ]:
scans = tp.load_scans()

models = {}
if TERSOFF_CHECKPOINT:
    models["tersoff"] = tp.load_model(str(ROOT / TERSOFF_CHECKPOINT))[0]
else:
    models["tersoff (priors)"] = tp.build_prior_tersoff_model()[0]
if WITH_REBO:
    models["tersoff:rebo (priors)"] = tp.build_prior_tersoff_model(tersoff_saturation="rebo")[0]
if PAIRING_CHECKPOINT:
    models["pairing"] = tp.load_model(str(ROOT / PAIRING_CHECKPOINT))[0]
else:
    models["pairing (priors)"] = tp.build_prior_model(ROOT / "configs" / "water_pairing.yaml")[0]

results = tp.evaluate_all(models, scans)
{name: sorted(res) for name, res in results.items()}

## The dimer: the discriminator

Left: the dimer's energy along O-O. Middle: the hydrogen bond's *raw* order `b = clip(J/κ)` (what
the rule has to squeeze -- about one at 2.5 Å with the pairing priors), the saturated `p`, and the
co-membership `c` the classical channels see. Right: the donor's covalent order and the donated
hydrogen's total. `waterfill` keeps the covalent bond at one and squeezes the hydrogen bond below
1e-4 like the solve; `rebo` shares linearly and the covalent bond drops to 0.2-0.5.

In [ ]:
r_oo = np.linspace(2.5, 4.0, 31)
dimer = tp.dimer_scan(models, r_oo)
tp.plot_dimer(r_oo, dimer, path="dimer");

## The O-H stretches and the bend

As in the pairing notebook: energies against Q-Chem, model minus Q-Chem with the mean removed, and
the one-body pieces. The `tersoff` and `pairing` curves should coincide on these.

In [ ]:
tp.plot_stretches(scans, results, path="stretches");
tp.plot_stretches(scans, results, x_max=1.8, path="stretches_zoom");
tp.plot_bend(scans, results, path="bend");

## The shared proton

Each curve referenced to its own midpoint (shapes), the absolute offset from Q-Chem in the legend.
The `tersoff` panel shows the known gap of the per-atom rule: a cusp within ~0.15 Å of the
midpoint where the shared proton's two orders sum to less than one.

In [ ]:
tp.plot_proton_transfer(scans, results, "h5o2+_pt", path="h5o2+_pt");
tp.plot_proton_transfer(scans, results, "h3o2-_pt", path="h3o2-_pt");

## The state along the proton transfer

One row per bond-order model: the shared proton's two orders and their sum; the oxygens' formal
charges (the solve's for `pairing`, the closed-form hand-over for `tersoff`); their capacities
against their coordination.

In [ ]:
tp.plot_pt_state(scans, results, "h5o2+_pt", path="h5o2+_pt_state");
tp.plot_pt_state(scans, results, "h3o2-_pt", path="h3o2-_pt_state");

## Bond orders, formal charges and capacities along every scan

In [ ]:
tp.plot_bond_orders(scans, results, path="bond_orders");
tp.plot_formal_charges(scans, results, path="formal_charges");
tp.plot_valence(scans, results, path="valence");

## The monomer AIMD sets

Fragment-energy parity on the thermal monomer frames; bias removed per model and quoted.

In [ ]:
parity = tp.monomer_parity(models, n_frames=MONOMER_FRAMES)
tp.plot_monomer_parity(parity, path="monomer_parity");